In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install wandb

import numpy as np
import torch
import torch.nn as nn
import torch.fft
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

wandb.login()

paths = {
    "G1_NURBS": (
        "/content/drive/MyDrive/LDC Dataset/nurbs_lid_driven_cavity_X.npz",
        "/content/drive/MyDrive/LDC Dataset/nurbs_lid_driven_cavity_Y.npz"
    ),
    "G2_Harmonics": (
        "/content/drive/MyDrive/LDC Dataset/harmonics_lid_driven_cavity_X.npz",
        "/content/drive/MyDrive/LDC Dataset/harmonics_lid_driven_cavity_Y.npz"
    ),
    "G3_Skeleton": (
        "/content/drive/MyDrive/LDC Dataset/skelneton_lid_driven_cavity_X.npz",
        "/content/drive/MyDrive/LDC Dataset/skelneton_lid_driven_cavity_Y.npz"
    )
}

In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.modes = modes
        self.scale = 1 / (in_channels * out_channels)
        self.weights = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, modes, modes, dtype=torch.cfloat)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = torch.fft.rfft2(x)
        out_ft = torch.zeros(
            B, self.weights.shape[1], H, W // 2 + 1,
            dtype=torch.cfloat, device=x.device
        )
        out_ft[:, :, :self.modes, :self.modes] = torch.einsum(
            "bixy,ioxy->boxy",
            x_ft[:, :, :self.modes, :self.modes],
            self.weights
        )
        return torch.fft.irfft2(out_ft, s=(H, W))


class FNO2d(nn.Module):
    def __init__(self, in_channels=3, modes=16, width=64):
        """
        FIX 1 — Parameterize in_channels instead of hardcoding 3.
        This makes the model reusable if input channels ever change.
        """
        super().__init__()
        self.fc0 = nn.Conv2d(in_channels, width, 1)

        self.conv1 = SpectralConv2d(width, width, modes)
        self.conv2 = SpectralConv2d(width, width, modes)
        self.conv3 = SpectralConv2d(width, width, modes)
        self.conv4 = SpectralConv2d(width, width, modes)

        self.w1 = nn.Conv2d(width, width, 1)
        self.w2 = nn.Conv2d(width, width, 1)
        self.w3 = nn.Conv2d(width, width, 1)
        self.w4 = nn.Conv2d(width, width, 1)

        self.fc1 = nn.Conv2d(width, 128, 1)
        self.fc2 = nn.Conv2d(128, 3, 1)

        self.act = nn.GELU()

    def forward(self, x):
        x = self.fc0(x)
        x = self.act(self.conv1(x) + self.w1(x))
        x = self.act(self.conv2(x) + self.w2(x))
        x = self.act(self.conv3(x) + self.w3(x))
        x = self.act(self.conv4(x) + self.w4(x))
        x = self.act(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
def relative_l2(pred, true):
    return torch.norm(pred - true) / (torch.norm(true) + 1e-8)


class Normalizer:
    """
    FIX 2 — Z-score normalization was completely absent in the original.
    FNO operates on raw field values directly, which can have very different
    scales across u, v, p — making loss gradients uneven and slowing convergence.
    Normalizing per-channel fixes this with minimal code overhead.
    """
    def __init__(self, data, eps=1e-8):
        # data: [N, C, H, W] — stats computed per channel
        self.mean = data.mean(dim=(0, 2, 3), keepdim=True)  # [1, C, 1, 1]
        self.std  = data.std(dim=(0, 2, 3), keepdim=True).clamp(min=eps)

    def encode(self, x):
        return (x - self.mean.to(x.device)) / self.std.to(x.device)

    def decode(self, x):
        return x * self.std.to(x.device) + self.mean.to(x.device)

In [ ]:
def train_fno_geometry(X_data, Y_data, geometry_name):
    print(f"\n===== Training FNO on {geometry_name} =====")

    # ── Downsample to 128x128 ─────────────────────────────────────────────────
    X_data = X_data[:, :, ::4, ::4]
    Y_data = Y_data[:, 0:3, ::4, ::4]

    X = torch.tensor(X_data, dtype=torch.float32)
    Y = torch.tensor(Y_data, dtype=torch.float32)

    # ── Train / Val split ─────────────────────────────────────────────────────
    X_train, X_val, Y_train, Y_val = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )

    # ── Fit normalizers on TRAINING data only ─────────────────────────────────
    x_norm = Normalizer(X_train)
    y_norm = Normalizer(Y_train)

    X_train = x_norm.encode(X_train).to(device)
    X_val   = x_norm.encode(X_val).to(device)
    Y_train_norm = y_norm.encode(Y_train).to(device)
    Y_val_norm   = y_norm.encode(Y_val).to(device)
    Y_val        = Y_val.to(device)   # raw, for relative L2 in physical space

    # ── DataLoader ────────────────────────────────────────────────────────────
    # FIX 3 — batch_size 4 → 8: GPU memory allows it at 128x128 and it
    # gives better gradient estimates, speeding up convergence.
    train_loader = DataLoader(
        TensorDataset(X_train, Y_train_norm),
        batch_size=8, shuffle=True
    )

    # ── Model & optimizer ─────────────────────────────────────────────────────
    in_channels = X_train.shape[1]
    model     = FNO2d(in_channels=in_channels, modes=16, width=64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-6)
    epochs    = 100

    # FIX 4 — Add cosine LR schedule: the original had no scheduler so LR
    # stayed fixed at 5e-4 for all 100 epochs, which is too large late in
    # training and prevents fine convergence.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )

    mse          = nn.MSELoss()
    best_val_l2  = float("inf")
    save_path    = f"/content/drive/MyDrive/LDC Dataset/{geometry_name}_fno_best.pt"

    # ── W&B init ──────────────────────────────────────────────────────────────
    wandb.init(
        project="SciML-FNO-Geometry-Comparison",
        name=f"FNO_{geometry_name}",
        config={
            "epochs":       epochs,
            "lr":           5e-4,
            "batch_size":   8,
            "modes":        16,
            "width":        64,
            "geometry":     geometry_name,
            "scheduler":    "CosineAnnealing",
            "normalization": "z-score per-channel",
            "in_channels":  in_channels,
        }
    )

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_num, train_den = 0.0, 0.0

        for xb, yb in train_loader:
            pred = model(xb)
            loss = mse(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_num  += torch.norm(pred - yb).item()
            train_den  += torch.norm(yb).item()

        scheduler.step()
        train_loss /= len(train_loader)
        train_l2    = train_num / (train_den + 1e-8)

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            pred_val_norm = model(X_val)                        # normalized space
            val_loss      = mse(pred_val_norm, Y_val_norm).item()
            val_l2_norm   = relative_l2(pred_val_norm, Y_val_norm).item()

            # Relative L2 in physical (denormalized) space for interpretability
            pred_val_raw  = y_norm.decode(pred_val_norm)
            val_l2_phys   = relative_l2(pred_val_raw, Y_val).item()

        if val_l2_phys < best_val_l2:
            best_val_l2 = val_l2_phys
            torch.save({
                "epoch":       epoch,
                "model":       model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_l2":      best_val_l2,
                "x_norm_mean": x_norm.mean,
                "x_norm_std":  x_norm.std,
                "y_norm_mean": y_norm.mean,
                "y_norm_std":  y_norm.std,
                "in_channels": in_channels,
            }, save_path)

        wandb.log({
            "epoch":           epoch,
            "train_loss":      train_loss,
            "train_rel_L2":    train_l2,
            "val_loss":        val_loss,
            "val_rel_L2_norm": val_l2_norm,
            "val_rel_L2_phys": val_l2_phys,   # physical space — most meaningful
            "best_val_L2":     best_val_l2,
            "lr":              scheduler.get_last_lr()[0],
        })

        if epoch % 10 == 0:
            print(f"{geometry_name} | Epoch {epoch:3d} | "
                  f"Train L2: {train_l2:.4f} | Val L2 (phys): {val_l2_phys:.4f} | "
                  f"Best: {best_val_l2:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    print(f"\n{geometry_name} done. Best Val L2 (physical): {best_val_l2:.4f}")
    wandb.finish()

    return {
        "geometry":         geometry_name,
        "final_train_loss": train_loss,
        "final_val_loss":   val_loss,
        "best_val_rel_L2":  best_val_l2,
    }

In [ ]:
results_fno = []

for name, (x_path, y_path) in paths.items():
    X_data = np.load(x_path)["data"]
    Y_data = np.load(y_path)["data"]
    metrics = train_fno_geometry(X_data, Y_data, name)
    results_fno.append(metrics)

df_fno = pd.DataFrame(results_fno)
print("\n===== FNO Comparison Table =====")
print(df_fno)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

def visualize_fno_predictions(geometry_name, model, y_norm, X_val_norm, Y_val_raw, H, W, device, num_samples=3):
    """
    Generates field comparison plots (U, V, P) and streamline plots
    for num_samples random validation samples. Logs all figures to W&B.
    """
    model.eval()
    indices = np.random.choice(len(X_val_norm), size=num_samples, replace=False)

    for idx in indices:
        xb = X_val_norm[idx:idx+1].to(device)   # [1, C, H, W], normalized
        yb = Y_val_raw[idx:idx+1].to(device)     # [1, 3, H, W], raw scale

        with torch.no_grad():
            pred_norm = model(xb)                # [1, 3, H, W]

        pred_raw = y_norm.decode(pred_norm)      # [1, 3, H, W]

        pred_np = pred_raw[0].cpu().numpy()      # [3, H, W]
        true_np = yb[0].cpu().numpy()            # [3, H, W]

        U_pred, V_pred, P_pred = pred_np[0], pred_np[1], pred_np[2]
        U_true, V_true, P_true = true_np[0], true_np[1], true_np[2]

        speed_pred = np.sqrt(U_pred**2 + V_pred**2)
        speed_true = np.sqrt(U_true**2 + V_true**2)

        x_lin = np.linspace(0, 1, W)
        y_lin = np.linspace(0, 1, H)
        X_grid, Y_grid = np.meshgrid(x_lin, y_lin)

        # ── Figure: 4 rows x 3 cols ───────────────────────────────────────────
        fig, axes = plt.subplots(4, 3, figsize=(18, 22))
        fig.suptitle(
            f"{geometry_name} — Sample {idx}\n"
            f"Top→Bottom: U velocity | V velocity | Pressure | Streamlines",
            fontsize=14, fontweight='bold', y=0.98
        )

        fields = [
            ("U Velocity", U_true, U_pred, "RdBu_r"),
            ("V Velocity", V_true, V_pred, "RdBu_r"),
            ("Pressure",   P_true, P_pred, "viridis"),
        ]

        # ── Rows 0-2: scalar field comparisons ───────────────────────────────
        for row, (label, true_f, pred_f, cmap) in enumerate(fields):
            error_f = np.abs(pred_f - true_f)
            vmin = min(true_f.min(), pred_f.min())
            vmax = max(true_f.max(), pred_f.max())

            im0 = axes[row, 0].imshow(true_f, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
            axes[row, 0].set_title(f"{label} — Ground Truth", fontsize=11)
            plt.colorbar(im0, ax=axes[row, 0], fraction=0.046, pad=0.04)

            im1 = axes[row, 1].imshow(pred_f, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
            axes[row, 1].set_title(f"{label} — Predicted", fontsize=11)
            plt.colorbar(im1, ax=axes[row, 1], fraction=0.046, pad=0.04)

            im2 = axes[row, 2].imshow(error_f, origin='lower', cmap='hot_r', aspect='equal')
            axes[row, 2].set_title(f"{label} — |Error|  (max={error_f.max():.3e})", fontsize=11)
            plt.colorbar(im2, ax=axes[row, 2], fraction=0.046, pad=0.04)

            for ax in axes[row]:
                ax.set_xticks([]); ax.set_yticks([])

        # ── Row 3: streamlines ────────────────────────────────────────────────
        streamline_configs = [
            ("Ground Truth Streamlines", U_true, V_true, speed_true),
            ("Predicted Streamlines",    U_pred, V_pred, speed_pred),
            ("Speed Error |‖u‖ - ‖û‖|", None,   None,   np.abs(speed_true - speed_pred)),
        ]

        for col, (title, U, V, speed) in enumerate(streamline_configs):
            ax = axes[3, col]
            if col < 2:
                strm = ax.streamplot(
                    X_grid, Y_grid, U, V,
                    color=speed, cmap='plasma', linewidth=1.2,
                    density=1.8, arrowsize=1.2,
                    norm=mcolors.Normalize(vmin=speed.min(), vmax=speed.max())
                )
                plt.colorbar(strm.lines, ax=ax, fraction=0.046, pad=0.04, label='Speed')
                ax.axhline(y=1.0, color='red', linewidth=2.0, linestyle='--', label='Lid')
                ax.legend(fontsize=8, loc='upper right')
            else:
                im = ax.imshow(speed, origin='lower', cmap='hot_r', aspect='equal')
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Speed Error')

            ax.set_title(title, fontsize=11)
            ax.set_xlabel("x"); ax.set_ylabel("y")

        plt.tight_layout(rect=[0, 0, 1, 0.97])

        save_fig_path = f"/content/drive/MyDrive/LDC Dataset/{geometry_name}_fno_sample{idx}.png"
        plt.savefig(save_fig_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_fig_path}")

        wandb.log({f"{geometry_name}/streamline_sample_{idx}": wandb.Image(fig)})
        plt.show()
        plt.close(fig)


def run_fno_visualization(geometry_name, x_path, y_path, checkpoint_path, device, num_samples=3):
    print(f"\n==== Visualizing {geometry_name} ====")

    X_data = np.load(x_path)["data"][:, :, ::4, ::4]
    Y_data = np.load(y_path)["data"][:, 0:3, ::4, ::4]

    X = torch.tensor(X_data, dtype=torch.float32)
    Y = torch.tensor(Y_data, dtype=torch.float32)

    _, X_val, _, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

    H, W = X_val.shape[2], X_val.shape[3]

    ckpt = torch.load(checkpoint_path, map_location=device)

    class RestoredNormalizer:
        def __init__(self, mean, std):
            self.mean = mean
            self.std  = std
        def encode(self, x):
            return (x - self.mean.to(x.device)) / self.std.to(x.device)
        def decode(self, x):
            return x * self.std.to(x.device) + self.mean.to(x.device)

    x_norm = RestoredNormalizer(ckpt["x_norm_mean"], ckpt["x_norm_std"])
    y_norm = RestoredNormalizer(ckpt["y_norm_mean"], ckpt["y_norm_std"])

    in_channels = ckpt["in_channels"]
    model = FNO2d(in_channels=in_channels, modes=16, width=64).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    X_val_norm = x_norm.encode(X_val)

    wandb.init(
        project="SciML-FNO-Geometry-Comparison",
        group="FNO-VIZ",
        name=f"FNO_{geometry_name}_viz"
    )

    visualize_fno_predictions(
        geometry_name=geometry_name,
        model=model,
        y_norm=y_norm,
        X_val_norm=X_val_norm,
        Y_val_raw=Y_val,
        H=H, W=W,
        device=device,
        num_samples=num_samples
    )

    wandb.finish()


# ── Run for all geometries ────────────────────────────────────────────────────
for geometry_name, (x_path, y_path) in paths.items():
    checkpoint_path = f"/content/drive/MyDrive/LDC Dataset/{geometry_name}_fno_best.pt"
    run_fno_visualization(geometry_name, x_path, y_path, checkpoint_path, device, num_samples=3)